# Two ways to predict where a ship is going

This project forecasts a vessel's future position from its recent AIS
track, nearby traffic, and the surrounding coastline. It does so twice,
in two branches that differ in one fundamental choice: **how they treat
time.**

This notebook explains that choice, shows what each approach feeds the
model and what it asks the model to produce, and compares them on real
data. Read it before the code.

| | `main` | `irregular-sampling` |
|---|---|---|
| input timing | resampled to a uniform 15-minute grid | raw pings at their true times |
| missing data | interpolated | left as a longer Δt |
| output | position at +15/30/45/60 min | position at each of the next N observations |
| can run on a live stream? | not really | yes |

Everything downstream — mesh, GNN, transformer, sampling head, loss — is
shared. Only the treatment of time differs.

In [ ]:
import sys, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '.')
plt.rcParams['figure.dpi'] = 110

## 1. The problem, and why time is awkward

AIS transmitters report on their own schedule. Rate depends on speed,
turn rate, and equipment class, and reports are lost when a vessel is
out of receiver range. So a vessel's track is **not** a neat time
series — it is a ragged sequence of observations.

Let's measure that on real data rather than assume it.

In [ ]:
from coastline import DMA_BOUNDS
from ais_ingest import load_dma_ais_csv

AIS_PATH = "/scratch/jtb3sud/maritime/ais/aisdk-2026-08-25.csv"

ais_df = load_dma_ais_csv(AIS_PATH, DMA_BOUNDS, pd.Timestamp.min, pd.Timestamp.max)
print(f"{len(ais_df):,} records, {ais_df['mmsi'].nunique():,} vessels, one day")

d = ais_df.sort_values(['mmsi', 'timestamp']).copy()
d['dt_sec'] = d.groupby('mmsi')['timestamp'].diff().dt.total_seconds()
gaps = d.dropna(subset=['dt_sec']).query('dt_sec > 0')

per_vessel_median = gaps.groupby('mmsi')['dt_sec'].median()
print("\nper-vessel median ping interval:")
for q in (0.10, 0.25, 0.50, 0.75, 0.90):
    print(f"  p{int(q*100):>2}: {per_vessel_median.quantile(q):>7.1f} s")

That spread is the whole motivation. A typical vessel reports every ~40
seconds, but the 10th and 90th percentiles differ by roughly **28x**.
"The next 4 pings" therefore means something completely different
depending on which ship you are looking at — anywhere from under a
minute to nearly twenty.

Two defensible responses:

1. **Regularize it.** Snap everything onto a fixed grid so every step is
   the same duration. Simple, and it makes the model's job easier.
2. **Represent it.** Keep observations where they are and tell the model
   how much time elapsed. Harder, but it matches what a live system
   actually receives.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(np.log10(per_vessel_median.clip(lower=1)), bins=60, color='steelblue', alpha=0.8)
for sec, label in [(10, '10 s'), (60, '1 min'), (300, '5 min'), (900, '15 min')]:
    ax.axvline(np.log10(sec), color='crimson', ls='--', lw=1)
    ax.text(np.log10(sec), ax.get_ylim()[1]*0.92, label, rotation=90,
            va='top', ha='right', fontsize=8, color='crimson')
ax.set_xlabel('per-vessel median ping interval (log10 seconds)')
ax.set_ylabel('vessels')
ax.set_title('Reporting rates vary by orders of magnitude across the fleet')
plt.show()

## 2. What each approach actually consumes

The clearest way to see the difference is to take one real vessel's
pings and draw how each branch reads them.

In [ ]:
def sampling_timeline_figure(ping_times_sec, interval_minutes=15, figsize=(15, 6)):
    """Same raw AIS track, drawn as each approach consumes it."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, sharex=True)
    t = np.asarray(ping_times_sec, dtype=float)
    step = interval_minutes * 60
    bins = np.arange(np.floor(t.min()/step)*step, t.max()+step, step)

    ax1.vlines(t/60, 0.55, 0.95, color='0.6', lw=1, label='raw AIS pings')
    ax1.vlines(bins/60, 0.05, 0.45, color='tab:blue', lw=2,
               label=f'{interval_minutes}-min bins (model input)')
    for b in bins:
        near = t[(t >= b - step) & (t < b)]
        if len(near):
            ax1.annotate('', xy=(b/60, 0.45), xytext=(near[-1]/60, 0.55),
                         arrowprops=dict(arrowstyle='->', color='tab:blue', alpha=0.4, lw=0.8))
    ax1.set_ylim(0, 1); ax1.set_yticks([])
    ax1.set_title('main: pings snapped to a uniform grid; empty bins interpolated', fontsize=11)
    ax1.legend(loc='upper right', fontsize=9)

    ax2.vlines(t/60, 0.55, 0.95, color='0.6', lw=1, label='raw AIS pings')
    ax2.vlines(t/60, 0.05, 0.45, color='tab:red', lw=2,
               label='model input = the pings themselves')
    ax2.set_ylim(0, 1); ax2.set_yticks([])
    ax2.set_xlabel('minutes')
    ax2.set_title('irregular-sampling: observations kept at true times; Δt is a model input',
                  fontsize=11)
    ax2.legend(loc='upper right', fontsize=9)
    fig.tight_layout()
    return fig


busy = ais_df['mmsi'].value_counts().index[5]
track = ais_df[ais_df.mmsi == busy].sort_values('timestamp').head(60)
t0 = track['timestamp'].iloc[0]
ping_sec = (track['timestamp'] - t0).dt.total_seconds().to_numpy()

fig = sampling_timeline_figure(ping_sec)
plt.show()

print(f"vessel {busy}: {len(ping_sec)} pings spanning {ping_sec.max()/60:.0f} minutes")
print(f"  gaps range {np.diff(ping_sec).min():.0f}s to {np.diff(ping_sec).max():.0f}s")

Two things are visible above.

**Bursts collapse.** Where several pings land inside one bin, `main`
keeps one and discards the rest. Information is thrown away.

**Silence is invented.** Where a bin contains no ping at all, `main`
fills it by interpolation — a position the vessel never reported. The
model cannot tell an observation from a guess.

The irregular branch does neither: every row it sees is something a
ship actually transmitted, and a long gap simply appears as a large Δt.

## 3. Input context, side by side

Both branches build the same kind of graph per timestep — mesh nodes for
geography, vessel nodes for traffic — but the vessel feature vector and
the notion of "a snapshot" differ.

### `main` — 12 features per vessel
```
[lon, lat, sog, cog_sin, cog_cos, type_onehot(7)]
```
A snapshot is **one timestamp**, holding every vessel present. Because
the world at a timestamp does not depend on which vessel you are
forecasting, the same graph is shared by all ego vessels (an 856x memory
saving that made multi-day training possible).

### `irregular-sampling` — 14 features per vessel
```
[lon, lat, sog, cog_sin, cog_cos, dt_norm, staleness_norm, type_onehot(7)]
```
A snapshot is **one ego-vessel ping**. The ego row is its exact reported
state; every other vessel is its *most recent report before that
instant*, carrying a `staleness` feature saying how old that report is.
That mirrors what a live system knows: you never have a neighbour's
current position, only its last transmission.

The two extra features are the whole point:

- **`dt_norm`** — time since this vessel's own previous report. Without
  it, "moved 2 km in 40 seconds" and "moved 2 km in 6 minutes" are
  indistinguishable to the network.
- **`staleness_norm`** — how out-of-date a neighbour's position is. A
  ship seen 20 seconds ago and one seen 20 minutes ago should not carry
  equal weight.

Because snapshots are per-ego-vessel here, they cannot be shared — which
is why this branch uses more memory for the same amount of data.

## 4. Output context — what the model is asked for

Both emit a **distribution**: many sampled trajectories per forward
pass, not one point. And both predict a **residual** from the last known
position rather than an absolute coordinate — the single largest
improvement in this project, and the same trick GraphCast uses for
atmospheric state.

Where they differ is what "the future" means.

### `main`
Predicts position at **+15, +30, +45, +60 minutes**. Fixed, known in
advance, identical for every window. Targets are grid points, so some
are interpolated rather than observed.

Normalization: displacement ÷ a single global standard deviation. That
works only because every target is the same distance into the future.

### `irregular-sampling`
Predicts position at **the next N actual observations**, whenever those
happen to arrive. Targets are always real reported positions, never
interpolated — but each window has its own horizon.

That breaks single-scale normalization: a 40-second target and a
19-minute target are not comparable quantities. So targets are
normalized as **velocity** instead:

```
velocity   = (target_position - anchor_position) / Δt
normalized = velocity / velocity_scale
```

A vessel holding steady speed then produces the same normalized target
regardless of horizon, so the model only has to learn *departures* from
constant velocity. And because the sampling head is separately
conditioned on Δt, it can still widen its uncertainty for longer
horizons — the scale-invariance does not flatten the uncertainty
structure.

A practical consequence: the irregular model can be **queried at
arbitrary future times**, including ones never seen in training. The
fixed-interval model can only answer the four horizons it was built for.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.axhline(1, color='tab:blue', lw=1)
ax.axhline(0, color='tab:red', lw=1)

for k, m in enumerate([15, 30, 45, 60]):
    ax.plot(m, 1, 'o', color='tab:blue', ms=11)
    ax.annotate(f'+{m}m', (m, 1), textcoords='offset points', xytext=(0, 13),
                ha='center', fontsize=9, color='tab:blue')

rng = np.random.default_rng(1)
for row, gap in [(0.0, 900)]:
    dts = np.cumsum(rng.lognormal(np.log(gap), 0.5, size=4)) / 60
    for dt in dts:
        ax.plot(dt, row, 'o', color='tab:red', ms=11)
        ax.annotate(f'+{dt:.0f}m', (dt, row), textcoords='offset points', xytext=(0, -20),
                    ha='center', fontsize=9, color='tab:red')

ax.set_yticks([0, 1])
ax.set_yticklabels(['irregular\n(actual pings)', 'main\n(fixed grid)'])
ax.set_xlabel('minutes into the future')
ax.set_xlim(-4, 90)
ax.set_ylim(-0.6, 1.6)
ax.set_title('What each model is asked to predict, for one window')
plt.show()

## 5. Baselines — how to read any result here

This matters more than any single metric, and it is easy to get wrong.

**Persistence** assumes the vessel stays exactly where it is. Its error
is therefore just how far the ship actually moved. For a vessel under
way this is a very easy bar, and "beats the baseline 89% of the time"
against persistence sounds far more impressive than it is.

**Constant velocity** (dead reckoning) extrapolates the last observed
velocity in a straight line. Ships mostly travel straight, so this is
genuinely hard to beat — and failure analysis on this project found
turning is by far the dominant error driver (worst-vs-best window turn
angle ratio of 4.0), which is consistent with the model doing something
close to linear extrapolation itself.

`viz.evaluate` reports both. **Read `beats_const_velocity_pct`.** A model
that beats persistence but not constant velocity has not learned
anything a straight line does not already give you.

In [ ]:
def constant_velocity_prediction(context_xy, n_steps):
    if len(context_xy) < 2:
        return np.repeat(context_xy[-1:], n_steps, axis=0)
    v = context_xy[-1] - context_xy[-2]
    return np.stack([context_xy[-1] + v * (k + 1) for k in range(n_steps)])


ctx_demo = np.array([[10.00, 55.00], [10.02, 55.01], [10.04, 55.02], [10.06, 55.03]])
truth_demo = np.array([[10.08, 55.045], [10.10, 55.055], [10.115, 55.070], [10.125, 55.088]])
cv = constant_velocity_prediction(ctx_demo, 4)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(*ctx_demo.T, 'ko-', label='observed context')
ax.plot(*np.vstack([ctx_demo[-1:], truth_demo]).T, 'r*-', ms=11, label='actual future')
ax.plot(*np.vstack([ctx_demo[-1:], cv]).T, 'b^--', label='constant velocity (dead reckoning)')
ax.plot(*np.repeat(ctx_demo[-1:], 2, axis=0).T, 'gs', ms=11, label='persistence ("stays put")')
ax.set_xlabel('lon'); ax.set_ylabel('lat')
ax.set_title('Persistence is easy to beat. Constant velocity is not.')
ax.legend(fontsize=9); ax.set_aspect('equal')
plt.show()

## 6. Results

Each branch writes a metrics JSON (see `export_metrics.py`, run once per
worktree). They cannot be imported into one process — both branches
define `model.py` and `graph_data.py` — so results are compared through
files.

**A fair comparison requires matched horizons.** `main` always predicts
60 minutes ahead. The irregular branch predicts wherever the pings land,
so its `--min-ping-gap-sec` must be tuned until its median horizon is
also ~60 minutes. Comparing a 60-minute forecast against a 14-minute one
says nothing.

In [ ]:
def load_metrics(path):
    if not os.path.exists(path):
        print(f"  (missing {path} -- run export_metrics.py in that worktree)")
        return None
    with open(path) as f:
        return json.load(f)


main_m = load_metrics('results/main_metrics.json')
irr_m = load_metrics('results/irregular_metrics.json')

if main_m and irr_m:
    rows = []
    for label, m in [('main (fixed 15-min)', main_m), ('irregular (raw pings)', irr_m)]:
        rows.append({
            'approach': label,
            'median horizon (min)': m.get('median_horizon_min', 60.0),
            'held-out windows': m.get('n_windows'),
            'spread correlation': m.get('spread_correlation'),
            'error (km)': m.get('model_mean_error_km'),
            'persistence (km)': m.get('persistence_error_km'),
            'beats persistence %': m.get('beats_persistence_pct'),
            'const-velocity (km)': m.get('const_velocity_error_km'),
            'beats const-velocity %': m.get('beats_const_velocity_pct'),
        })
    display(pd.DataFrame(rows).set_index('approach').T.round(3))

### Reading the table

- **spread correlation** — does the model's own uncertainty track how
  far the vessel actually moved? This is what the residual + normalized
  target work was for; near zero means the samples are decorative.
- **beats const-velocity %** — the real accuracy claim.
- **median horizon** — if these differ between rows, the error columns
  are not comparable and nothing else in the table means much.

## 7. When each one is the right choice

**Use `main` when** you are working offline on historical archives,
want fixed reporting horizons, and want the simplest thing that works.
Regularizing the input genuinely makes the learning problem easier, and
it has been ahead on accuracy so far.

**Use `irregular-sampling` when** you want to run on a live AIS feed.
The fixed-interval model structurally cannot: it would have to wait for
bin boundaries, interpolate positions it has not observed, and it has no
way to express that a neighbour's position is twenty minutes stale. The
irregular model consumes pings as they arrive, and its KV-cached
`.step()` path is built exactly for that — verified to match full
recomputation to ~1e-6 with genuinely irregular timestamps.

The honest summary: regularizing buys accuracy, representing time buys
deployability. Which matters depends on whether you are writing a paper
or watching a harbour.

## 8. Where to look next

- `viz.py` / `viz_irregular.py` — evaluation, prediction plots,
  animations, and failure analysis for each branch
- `PROJECT_STATUS.md` — full history, every bug found and why it
  mattered, and current open questions
- **Known open issue:** batched training intermittently triggers a CUDA
  device-side assert; `CUDA_LAUNCH_BLOCKING=1` avoids it at a throughput
  cost. Unresolved.
- **Biggest untried idea:** AIS carries a Rate-of-Turn field that is
  currently dropped at ingest. Given that turning is the measured
  dominant failure mode, feeding it in is the obvious next experiment.